In [1]:
from rdflib import Graph, URIRef
from rdflib.namespace import RDF, RDFS, OWL, SKOS
from dash import Dash, html, dcc, Input, Output, State, ctx
import dash_cytoscape as cyto
import json
from rdflib import Literal, Namespace
import random
import os
import re
import dash_draggable
import base64

In [2]:
g = Graph()
g.parse("ontology/HolyWells_Ontology.rdf")
print(f"{len(g)} triples loaded")

SCHEMA = Namespace("http://schema.org/")

797 triples loaded


In [3]:
def label(resource):
    """
    Return a human-readable label for a resource or literal.

    Priority:
      1. rdfs:label
      2. skos:altLabel
      3. URI fragment / last path segment
      4. literal value
    """

    # If a plain string was passed, convert URI-like strings to URIRef
    if isinstance(resource, str):
        if resource.startswith("http://") or resource.startswith("https://"):
            resource = URIRef(resource)
        else:
            return  resource

    # Literal
    if isinstance(resource, Literal):
        return str(resource)

    # rdfs:label
    lbl = g.value(resource, RDFS.label)
    if lbl is not None:
        return str(lbl)

    # skos:altLabel
    alt = g.value(resource, SKOS.altLabel)
    if alt is not None:
        return  str(alt)

    # Fallback to URI fragment
    text = str(resource)
#here
    if "#" in text:
        return text.rsplit("#", 1)[-1]

    return text.rstrip("/").rsplit("/", 1)[-1]

In [4]:
classes = set(g.subjects(RDF.type, OWL.Class))
print(f"Found {len(classes)} classes")

individuals = set()
for s, o in g.subject_objects(RDF.type):
    if o not in (OWL.Class, RDFS.Class):
        individuals.add(s)
print(f"Found {len(individuals)} individuals")

named_individuals = set(g.subjects(RDF.type, OWL.NamedIndividual))
print(f"Named individuals: {len(named_individuals)}")

unnamed_individuals = individuals - named_individuals
print(f"Unnamed individuals: {len(unnamed_individuals)}")

Found 108 classes
Found 173 individuals
Named individuals: 96
Unnamed individuals: 77


In [5]:
# Convert resources to string sets for easy lookup
classes_string = {(str(c) )for c in classes}

individuals_string = {str(i) for i in individuals}

label_literals = {
    str(lit)
    for _, lit in g.subject_objects(RDFS.label)
    if isinstance(lit, Literal)
}

alt_label_literals = {
    str(lit)
    for _, lit in g.subject_objects(SKOS.altLabel)
    if isinstance(lit, Literal)
}

description_literals = {
    str(lit)
    for _, lit in g.subject_objects(SCHEMA.description)
    if isinstance(lit, Literal)
}

all_literals = {
    str(obj)
    for _, _, obj in g
    if isinstance(obj, Literal)
}

other_literals = (
    all_literals
    - label_literals
    - alt_label_literals
    - description_literals
)

In [6]:
def get_node_kind(value):
    """
    Classify a node value (URI string or literal string).
    """

    # Choose symbols
    CLASS_SYMBOL = "● "         #black circle
    INDIVIDUAL_SYMBOL = "◆ "    #black diamond
    LITERAL_SYMBOL = "▲ "       #black triangle
    LABEL_SYMBOL =    "■ "      #black square
    ALT_LABEL_SYMBOL= "⬢ "      #black hexagon
    DESCRIPTION_SYMBOL= "⬟ "     #black pentagon
    
    # Try to interpret the value as a URI resource
    try:
        uri = str(URIRef(value))
    except Exception:
        uri = None
    # ----- Literal categories -----

    if value in label_literals:
        return ["label", LABEL_SYMBOL]

    if value in alt_label_literals:
        return ["alt_label", ALT_LABEL_SYMBOL]

    if value in description_literals:
        return ["description", DESCRIPTION_SYMBOL]

    if value in other_literals:
        return ["literal", LITERAL_SYMBOL]

    # ----- Resource categories -----

    if value in classes_string:
        return ["class", CLASS_SYMBOL]

    if value in individuals_string:
        return ["individual", INDIVIDUAL_SYMBOL]

    return ["resource", ""]

In [7]:
def get_predicates(resource):
    subject = URIRef(resource)

    outgoing = sorted({
        str(predicate)
        for predicate in g.predicates(subject, None)
    })

    incoming = sorted({
        str(predicate)
        for predicate in g.predicates(None, subject)
    })

    return {
        "outgoing": outgoing,
        "incoming": incoming,
    }

In [8]:
get_predicates("http://ontology.holywells.link/ontology/YouTube_Video_kxRnXwX0HAM")

{'outgoing': ['http://www.cidoc-crm.org/cidoc-crm/P1_is_identified_by',
  'http://www.cidoc-crm.org/cidoc-crm/P43_has_dimension',
  'http://www.cidoc-crm.org/cidoc-crm/P67_refers_to',
  'http://www.w3.org/1999/02/22-rdf-syntax-ns#type'],
 'incoming': ['http://www.cidoc-crm.org/cidoc-crm/P92_brought_into_existence']}

What I still want:

1. expand all option for each node
2. delete orphan nodes by default

5. export image as svg and png


In [9]:
# Dropdown options and nodes types
all_nodes_options =[]
classes_options = []
individuals_options = []
named_individuals_options =[]
unnamed_individuals_options =[]

# set up the options for the drop down menus, keep all options. 
for subject in set(g.subjects()):

    # Only URI resources - no blanks
    if not isinstance(subject, URIRef):
        continue

    all_nodes_options.append(
        {
            "label": f"{label(subject)}",
            "value": str(subject)
        }
    )

    if (subject in classes):
        classes_options.append(
            {
                "label": f"{label(subject)}",
                "value": str(subject)
            }
        )
    if (subject in named_individuals):
        named_individuals_options.append(
            {
                "label": f"{label(subject)}",
                "value": str(subject)
            }
        )

    if (subject in unnamed_individuals):
        unnamed_individuals_options.append(
            {
                "label": f"{label(subject)}",
                "value": str(subject)
            }
        )
#sort options alphabetically
classes_options.sort(key=lambda d:d["label"])
named_individuals_options.sort(key=lambda d:d["label"])
unnamed_individuals_options.sort(key=lambda d:d["label"])
all_nodes_options.sort(key= lambda d:d["label"])

type_options =["Class", "Named Individual", "Unnamed Individual"]


all_options = {'Class':classes_options, 'Named Individual':named_individuals_options, 'Unnamed Individual':unnamed_individuals_options}

In [10]:
# Helpers
# ============================================================

def hidden_delete_style():
    return {
        "display": "none",
        "position": "absolute",
        "zIndex": 1001,
    }


def hidden_menu_style():
    return {
        "display": "none",
    }


def node_controls_style(x, y):
    return {
        "display": "block",
        "position": "absolute",
        "left": f"{x + 15}px",
        "top": f"{y + 15}px",
        "zIndex": 1000,
        "pointerEvents": "auto",
    }


def delete_button_style(x, y):
    return {
        "display": "block",
        "position": "absolute",
        "left": f"{x + 15}px",
        "top": f"{y - 40}px",
        "zIndex": 1001,
        "pointerEvents": "auto",
    }

In [11]:
# Dash application style

app = Dash(__name__)

# ----------------------------------------------------
# Floating legend (put this BEFORE app.layout)
# ----------------------------------------------------
def legend_row(symbol, text, color):
    return html.Div(
        [
            html.Span(symbol, style={"fontWeight": "bold", "marginRight": "8px"}),
            html.Span(text),
        ],
        style={
            "backgroundColor": color,
            "borderRadius": "8px",
            "padding": "6px 10px",
            "marginBottom": "6px",
            "color": "black",
            "border": "1px solid rgba(0,0,0,0.15)",
        },
    )

legend = html.Div(
    [
        html.Div(
            "Legend",
            style={
                "fontWeight": "bold",
                "marginBottom": "10px",
                "padding": "8px",
                "backgroundColor": "#f0f0f0",
                "borderRadius": "6px",
                "textAlign": "center",
            },
        ),
        legend_row("●", "Class", "#FAC369"),
        legend_row("◆", "Individual", "#BCA7C7"),
        legend_row("■", "Label", "#B38D68"),
        legend_row("⬢", "Alternative label", "#ED9F77"),
        legend_row("⬟", "Description", "#F9BC90"),
        legend_row("▲", "Literal", "#95DEA0"),
    ],
    id="legend-box",
    style={
        "position": "absolute",
        "left": "20px",
        "top": "20px",
        "zIndex": 3000,
        "backgroundColor": "white",
        "border": "1px solid #888",
        "borderRadius": "10px",
        "padding": "10px",
        "boxShadow": "2px 2px 10px rgba(0,0,0,0.25)",
        "width": "190px",
        "cursor": "move",
        "userSelect": "none",
    },
)

app.layout = html.Div(

    [

        html.H2("RDF Graph Browser"),
    
        # ----------------------------------------------------
        # Node Type Selector
        # ----------------------------------------------------

        dcc.Dropdown(

            id="type-dropdown",
            
            options=type_options,

            placeholder="Choose Type",

            searchable=True,

            clearable=True,

            style={
                "width":"60%"
            }

        ),

        html.Br(),

        # ----------------------------------------------------
        # Node Selector
        # ----------------------------------------------------

        dcc.Dropdown(

            id="node-dropdown",
            
            options=all_nodes_options,

            placeholder="Choose starting node",

            searchable=True,

            clearable=True,

            style={
                "width":"60%"
            }

        ),

        html.Br(),

        # ----------------------------------------------------
        # Save and upload button
        # ----------------------------------------------------
        dcc.Input(
            id="save-filename",
            type="text",
            placeholder="graph_name.json",
            value="my_graph",
            style={"width": "250px", "marginRight": "10px"}
        ),

        html.Button( "Save as...", id="save-as-button"),

        html.Button("Load graph", id="load-graph", style={"marginLeft": "10px"}),

         html.Button("Export PNG", id="export-png-button", style={"marginLeft": "10px"}
        ),

        html.Div(id="save-status", style={"marginTop": "10px"}),

       
        html.Br(),

        # ----------------------------------------------------
        # Edge Expansion Selector, draggable legend and graph
        # under same div so selector is placed relative to node
        # ----------------------------------------------------
        html.Div(
            [
        

            
                # ----------------------------------------------------
                # Cytoscape graph style
                # ----------------------------------------------------

                cyto.Cytoscape(

                    id="graph",
                    elements=[],
                    layout={
                        "name":"preset",
                        "animate":True
                    },
                    style={
                        "width":"100%",
                        "height":"900px"
                    },

                    stylesheet=[
                        # --------------------------------------------
                        # Nodes
                        # --------------------------------------------
                        {
                            "selector":"node[kind='class']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#FAC369", #yellow
                                "color": "black",
                                "shape": "round-rectangle",
                            
                                "width": "label",
                                "height": "label",

                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 12,

                            }

                        },

                        {
                            "selector":"node[kind='individual']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#BCA7C7", #"#6D3D80", # purple
                                "color": "black",

                                "shape": "round-rectangle",

                                "width": "label",
                                "height": "label",

                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 12,
                            }

                        },

                        {
                            "selector":"node[kind='label']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#B38D68", # orange
                                "color": "black",

                                "shape": "round-rectangle",

                                "width": "label",
                                "height": "label",

                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 12

                            }

                        },
                        {
                            "selector":"node[kind='alt_label']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#ED9F77", # tangerine
                                "color": "black",

                                "shape": "round-rectangle",

                                "width": "label",
                                "height": "label",

                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 12

                            }

                        },


                        {
                            "selector":"node[kind='description']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#F9BC90", # peach
                                "color": "black",

                                "shape": "round-rectangle",

                                "width": "label",
                                "height": "label",
                                
                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 12

                            }

                        },

                        {
                            "selector":"node[kind='literal']",

                            "style":
                            {
                                "label": "data(label)",
                                "background-color": "#95DEA0", # green 
                                "color": "black",

                                "shape": "round-rectangle",

                                "width": "label",
                                "height": "label",

                                "padding": "10px",
                                "padding-left": "24px",
                                "padding-right": "10px",
                                "padding-top": "10px",
                                "padding-bottom": "10px",

                                "text-wrap": "wrap",
                                "text-max-width": "120px",

                                "text-valign": "center",
                                "text-halign": "center",

                                "font-size": 12

                            }

                        },

                        {
                            "selector": "edge[kind='rdf_edge']",
                            "style": {
                                "label": "data(label)",
                                "curve-style": "bezier",
                                "target-arrow-shape": "triangle",
                                "width": 2,
                                "font-size": 12,
                                "text-background-color": "white",
                                "text-background-opacity": 1
                            }
                        },
                    ]

                ),
            
                # =========================================================
                # Floating controls
                # =========================================================
                html.Div(
                    [
                        html.Button(
                            "Delete selected",
                            id="delete-button",
                            disabled=True,
                            style=hidden_delete_style(),
                            
                        ),

                        dcc.Dropdown(
                            id="predicate-menu",
                            options=[],
                            value=None,
                            placeholder="Expand on...",
                            style=hidden_menu_style(),
                        ),
                    ],

                    id="floating-controls",
                    style={
                        "position": "absolute",
                        "top": "0px",
                        "left": "0px",
                        "width": "100%",
                        "height": "100%",
                        "zIndex": 1000,
                        # The overlay itself does not intercept Cytoscape clicks.
                        "pointerEvents": "none",
                    },
                ),

                dcc.Store(id="clicked-resource"),
                dcc.Store(id="selected-element"),
                dcc.Store(id="debug-store"),
                dcc.Store(id="png-export-data"),


                legend,

            ],
            id="graph-export-container",
            style={
                "position": "relative",
                "width": "100%",
                "height": "900px",
            },
        )

    ]


)

In [12]:
# handle selection
@app.callback(
    Output("selected-element", "data"),

    Output("delete-button", "disabled"),
    Output("delete-button", "style"),

    Output("predicate-menu", "options"),
    Output("predicate-menu", "style"),
    Output("predicate-menu", "value"),

    Output("clicked-resource", "data"),

    Output("graph", "elements"), #allow_duplicate=True

    Output("debug-store", "data"),

    Input("graph", "tapNode"),
    Input("graph", "tapEdge"),
    Input("delete-button", "n_clicks"),
    Input("predicate-menu", "value"),


    State("selected-element", "data"),
    State("clicked-resource", "data"),
    State("graph", "elements"),

    #prevent_initial_call=True,
)
def handle_selection(
    node_data,
    edge_data,
    n_clicks,
    predicate_selection,
    selected,
    resource,
    elements,
):
    triggered = ctx.triggered_prop_ids

    debug_data = {
        "triggered": list(triggered.keys()),
        "node_data": node_data,
        "edge_data": edge_data,
        "selected_before": selected,
        "resource": resource,
        "predicate_selection": predicate_selection,
    }

    # ========================================================
    # DELETE
    # ========================================================

    if "delete-button.n_clicks" in triggered:

        if selected:

            selected_id = selected["id"]

            # --------------------------------------------
            # Delete selected node
            # --------------------------------------------

            if selected["type"] == "node":

                elements = [
                    e
                    for e in elements
                    if e["data"].get("id") != selected_id
                    and e["data"].get("source") != selected_id
                    and e["data"].get("target") != selected_id
                ]

            # --------------------------------------------
            # Delete selected edge
            # --------------------------------------------

            elif selected["type"] == "edge":

                elements = [
                    e
                    for e in elements
                    if e["data"].get("id") != selected_id
                ]

        # --------------------------------------------
        # Delete closes UI
        # --------------------------------------------

        return (
            None,                  # selected-element
            True,                  # delete-button.disabled
            hidden_delete_style(), # delete-button.style

            [],                    # predicate-menu.options
            hidden_menu_style(),   # predicate-menu.style
            None,                  # predicate-menu.value

            None,                  # clicked-resource

            elements,              # graph.elements

            debug_data,            # debug-store
        )

    # ========================================================
    # PREDICATE EXPANSION
    # ========================================================

    if "predicate-menu.value" in triggered:

        if predicate_selection and resource:

            direction, predicate = predicate_selection.split("::", 1)

            elements = add_single_predicate(
                resource=resource,
                predicate=predicate,
                direction=direction,
                elements=elements,
            )

        # --------------------------------------------
        # Expansion closes UI
        # --------------------------------------------

        return (
            None,                  # selected-element
            True,                  # delete-button.disabled
            hidden_delete_style(), # delete-button.style

            [],                    # predicate-menu.options
            hidden_menu_style(),   # predicate-menu.style
            None,                  # predicate-menu.value

            None,                  # clicked-resource

            elements,              # graph.elements

            debug_data,            # debug-store
        )

    # ========================================================
    # NODE CLICK
    # ========================================================

    if "graph.tapNode" in triggered and node_data:

        data = node_data["data"]
        pos = node_data["renderedPosition"]

        node_id = data["id"]

        # --------------------------------------------
        # Predicate options
        # --------------------------------------------

        predicates = get_predicates(node_id)

        options = []

        for predicate in predicates["outgoing"]:
            options.append(
                {
                    "label": f"→ {label(URIRef(predicate))}",
                    "value": f"out::{predicate}",
                }
            )

        for predicate in predicates["incoming"]:
            options.append(
                {
                    "label": f"← {label(URIRef(predicate))}",
                    "value": f"in::{predicate}",
                }
            )

        # --------------------------------------------
        # Selected node
        # --------------------------------------------

        selected_element = {
            "type": "node",
            "id": node_id,
        }

        return (
            selected_element,

            False,
            delete_button_style(
                pos["x"],
                pos["y"],
            ),

            options,
            node_controls_style(
                pos["x"],
                pos["y"],
            ),

            None,       # predicate-menu.value
            node_id,    # clicked-resource

            elements,

            debug_data,
        )

    # ========================================================
    # EDGE CLICK
    # ========================================================

    if "graph.tapEdge" in triggered and edge_data:

        edge_id = edge_data["data"]["id"]

        selected_element = {
            "type": "edge",
            "id": edge_id,
        }

        source_id = edge_data["data"]["source"]
        target_id = edge_data["data"]["target"]

        source_node = next(
            e for e in elements
            if e["data"].get("id") == source_id
        )

        target_node = next(
            e for e in elements
            if e["data"].get("id") == target_id
        )

        source_pos = source_node["position"]
        target_pos = target_node["position"]

        edge_x = (source_pos["x"] + target_pos["x"]) / 2
        edge_y = (source_pos["y"] + target_pos["y"]) / 2

        return (
            selected_element,

            False,
            delete_button_style(edge_x, edge_y),

            [],
            hidden_menu_style(),
            None,

            None,

            elements,

            debug_data,
        )

    # ========================================================
    # FALLBACK
    # ========================================================

    return (
        selected,

        selected is None,
        hidden_delete_style(),

        [],
        hidden_menu_style(),
        None,

        resource,

        elements,

        debug_data,
    )

In [13]:
# Callback Expand predicate

@app.callback(
    Output("graph", "elements", allow_duplicate=True),
    
    Input("predicate-menu", "value"),

    State("clicked-resource", "data"),
    State("graph", "elements"),

    prevent_initial_call=True,
)

def expand_selected_predicate(selection, resource, elements):

    if not selection or not resource:
        return (
            elements
        )

    direction, predicate = selection.split("::", 1)

    elements = add_single_predicate(
        resource=resource,
        predicate=predicate,
        direction=direction,
        elements=elements,
    )

    # ========================================================
    # Expansion is complete:
    # clear all selection/UI state
    # ========================================================
    
    return (
        elements
    )

def add_node(elements, node_id, label_text, kind):

    elements.append({
        "data": {
            "id": node_id,
            "label": label_text,
            "kind": kind
        },
       
        "position": {
            "x": random.randint(0, 800),
            "y": random.randint(0, 600)
        }
    })


def add_single_predicate(resource, predicate, direction, elements):
    resource_ref = URIRef(resource)

    existing_nodes = {
        e["data"]["id"]
        for e in elements
        if (
        "id" in e["data"]
    )
    }

    existing_edges = {
        (
            e["data"].get("source"),
            e["data"].get("target"),
            e["data"].get("predicate")
        )
        for e in elements
        if "source" in e["data"]
    }

    if direction == "out":
        triples = [
            (resource, str(obj))
            for obj in g.objects(resource_ref, URIRef(predicate))
        ]
    else:
        triples = [
            (str(subj), resource)
            for subj in g.subjects(URIRef(predicate), resource_ref)
        ]

    for source, target in triples:
        if target not in existing_nodes:
            add_node(
                elements,
                target,
                get_node_kind(target)[1]+label(URIRef(target)),
                get_node_kind(target)[0],
            )
            existing_nodes.add(target)

        if source not in existing_nodes:
            add_node(
                elements,
                source,
                get_node_kind(source)[1]+label(URIRef(source)),
                get_node_kind(source)[0],
            )
            existing_nodes.add(source)

        edge = (source, target, predicate)

        if edge not in existing_edges:
            elements.append({

                "data": {
                    "id": f"{source}|{predicate}|{target}",
                    "source": source,
                    "target": target,
                    "predicate": predicate,
                    "label": label(URIRef(predicate)),
                    "kind": "rdf_edge"
                }
            })
            existing_edges.add(edge)
    

    return elements


In [14]:
# Callback Dropdown Type Picker

@app.callback(

   Output(
        "node-dropdown",
        "options"
    ),

    Input(
        "type-dropdown",
        "value"
    )

)

def set_node_value_options(selected_type):
    if(selected_type is None):
        return all_nodes_options
    return all_options[selected_type]




In [15]:
# Callback Dropdown Start Node Picker

@app.callback(

   Output("graph", "elements", allow_duplicate=True),

  Input("node-dropdown", "value"),  
  prevent_initial_call=True,

)

def create_start_graph(uri):
    if uri is None:
        return []

    elements = []

    add_node(
        elements,
        node_id=uri,
        label_text=get_node_kind(uri)[1] + label(URIRef(uri)),
        kind=get_node_kind(uri)[0]
    )

    return elements
    
def load_start_node(uri):
    if uri is None:
        return []
    elements = create_start_graph(uri)
    return elements


In [16]:
# Callback Save Layout

@app.callback(
    Output("save-status", "children"),
    Input("save-as-button", "n_clicks"),
    State("save-filename", "value"),
    State("graph", "elements"),
    State("graph", "pan"),
    State("graph", "zoom"),
    prevent_initial_call=True,
)
def save_graph_as(n_clicks, filename, elements, pan, zoom):

    if not filename:
        return "Please enter a filename."

    # Remove illegal filename characters
    filename = re.sub(r'[\\\\/:*?\"<>|]', "_", filename.strip())

    # Add .json 
    if not filename.lower().endswith(".json"):
        filename += ".json"

    # Create folder if needed
    os.makedirs("saved_graphs", exist_ok=True)

    path = os.path.join("saved_graphs", filename)

    state = {
        "elements": elements,
        "pan": pan,
        "zoom": zoom,
    }

    with open(path, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)

    return f"Saved graph to: {path}"


In [17]:
@app.callback(
    Output(
        "save-status",
        "children",
        allow_duplicate=True
    ),
    Input(
        "png-export-data",
        "data"
    ),
    State(
        "save-filename",
        "value"
    ),
    prevent_initial_call=True,
)
def save_png_file(image_data, filename):

    if not image_data:
        return "PNG export failed: no image data received."

    if not filename:
        return "Please enter a filename."

    # --------------------------------------------------------
    # Clean filename
    # --------------------------------------------------------

    filename = re.sub(
        r'[\\/:*?"<>|]',
        "_",
        filename.strip()
    )

    # Remove .json
    if filename.lower().endswith(".json"):
        filename = filename[:-5]

    # Remove .png
    if filename.lower().endswith(".png"):
        filename = filename[:-4]

    # Final filename
    png_filename = filename + ".png"


    # --------------------------------------------------------
    # Create output folder
    # --------------------------------------------------------

    os.makedirs(
        "saved_images",
        exist_ok=True
    )


    # --------------------------------------------------------
    # Full path
    # --------------------------------------------------------

    path = os.path.join(
        "saved_images",
        png_filename
    )


    # --------------------------------------------------------
    # Decode data URL
    # --------------------------------------------------------

    try:

        header, encoded = image_data.split(
            ",",
            1
        )

        image_bytes = base64.b64decode(
            encoded
        )

    except Exception as e:

        return f"PNG export failed: {e}"


    # --------------------------------------------------------
    # Write PNG
    # --------------------------------------------------------

    try:

        with open(
            path,
            "wb"
        ) as f:

            f.write(image_bytes)

    except Exception as e:

        return f"PNG export failed while saving: {e}"


    return f"Saved PNG to: {path}"

In [18]:
# Callback Load Saved Layout

@app.callback(
    Output("graph", "elements", allow_duplicate=True),
    Output("graph", "pan", allow_duplicate=True),
    Output("graph", "zoom", allow_duplicate=True),
    Input("load-graph", "n_clicks"),
    State("save-filename", "value"),
    prevent_initial_call=True,
)
def load_graph_named(n_clicks, filename):

    if not filename:
        raise dash.exceptions.PreventUpdate

    if not filename.lower().endswith(".json"):
        filename += ".json"

    path = os.path.join("saved_graphs", filename)

    if not os.path.exists(path):
        raise dash.exceptions.PreventUpdate

    with open(path, "r", encoding="utf-8") as f:
        state = json.load(f)

    return (
        state.get("elements", []),
        state.get("pan", {"x": 0, "y": 0}),
        state.get("zoom", 1),
    )

In [19]:
# RUN SERVER
# ============================================================


if __name__ == "__main__":

    app.run(
        debug=True
    )